# Phase 2: generate the training motions 🐾

Fills one dataset folder on Drive with gated motions for the general tracker, using `kflib`
(`final_project/kflib/`). The decisions are in `final_project/phase2_proposal.md`:
- ~250 Gemini motions, ~240 procedural and the course `sit`, each mirrored, a third also time-scaled;
- keep what the gate passes (plain-replay FALLS rejected);
- hold out `sneeze`, `push_up`, `stand_sit_shake_sit_stand` and the benchmark's 12 requests.

Every step can be stopped and run again: motions already in the dataset are skipped, nothing is
overwritten. A CPU runtime is enough. **Nothing here moves the robot.**

In [ ]:
# @title Setup: pupper-mjlab (pinned), kflib, Drive (about 2 minutes)
import os, sys
COMMIT = "aa475e9a393a209b29fd5de7e0ee10f3de643230"  # the pupper-mjlab commit Phase 0 read
BRANCH = "final-project-kflib"  # @param {type:"string"}
DATASET_NAME = "v1"  # @param {type:"string"}

if not os.path.exists("/content/pupper-mjlab"):
  !git clone -q https://github.com/cs123-stanford/pupper-mjlab /content/pupper-mjlab
!git -C /content/pupper-mjlab checkout -q {COMMIT}
!pip install -q -e /content/pupper-mjlab
sys.path.insert(0, "/content/pupper-mjlab/src")  # the running kernel can't see an editable install

if not os.path.exists("/content/gemini_pupper"):
  !git clone -q -b {BRANCH} https://github.com/MlsBli/gemini_pupper /content/gemini_pupper
!git -C /content/gemini_pupper pull -q
sys.path.insert(0, "/content/gemini_pupper/final_project")

from google.colab import drive
drive.mount("/content/drive")  # pick the Google account whose Drive should hold the dataset
from pathlib import Path
ROOT = Path("/content/drive/MyDrive/pupper_general_tracker")
DATASET = ROOT / "datasets" / DATASET_NAME
DATASET.mkdir(parents=True, exist_ok=True)
print("dataset:", DATASET, "| already has", len(list((DATASET / "motions").glob("*.npz"))) if (DATASET / "motions").exists() else 0, "motions")

In [ ]:
# @title Self-test (about 1 minute; should end "N/N checks passed", all passing)
%cd /content/gemini_pupper/final_project
!python -m kflib.selftest 2>/dev/null | tail -3

In [ ]:
# @title Fix the held-out split (once, before anything is generated or trained)
from kflib.dataset import Dataset, report
from kflib.course import HELD_OUT
from kflib.metrics import REQUESTS as BENCHMARK_REQUESTS
ds = Dataset(DATASET)
if not (DATASET / "split.json").exists():
  ds.write_split(dict(course_tricks=list(HELD_OUT),
                      benchmark_requests={r["id"]: r["text"] for r in BENCHMARK_REQUESTS},
                      note="4b: requests written after training starts are also held out"))
print((DATASET / "split.json").read_text())

In [ ]:
# @title The LLM (free Colab AI)
from kflib.llm import LLM
llm = LLM("colab")
print("model:", llm.model)
text, seconds, _ = llm("Reply with JSON only.", 'Reply with exactly {"ok": true}')
print(text.strip()[:60], f"({seconds:.1f} s)")

In [ ]:
# @title Write the requests (~300, so ~250 survive the motion gate; a few minutes)
import json
from collections import Counter
from kflib.request_gen import generate_requests
POOL = DATASET / "requests_pool.jsonl"
requests = generate_requests(llm, POOL, n_total=300)
rejected = [p for p in map(json.loads, POOL.read_text().splitlines()) if p.get("rejected")]
print(len(requests), "usable requests:", dict(Counter(r["category"] for r in requests)))
print(len(rejected), "filtered out:", dict(Counter(r["rejected"].split(":")[0] for r in rejected)))
for r in requests[::30]:
  print(f"  [{r['category']}] {r['text']}")

## Pilot first: 20 requests

Measures Colab AI's speed and rate limits before the long run. Look at: how many pass, how many
attempts they need, and the seconds per request. If it's much slower than the benchmark's ~16 s per
LLM call, or rate limits keep pausing it, consider AI Studio (`LLM("aistudio", "<model>")`, key in
Colab Secrets) before the full run.

In [ ]:
# @title Pilot: the first 20 requests, one at a time
import json, time, statistics
from kflib.generate import add_gemini
t0 = time.time()
res = add_gemini(ds, requests[:20], llm, threads=1)
log = [json.loads(l) for l in (DATASET / "requests.jsonl").read_text().splitlines()]
pilot = [e for e in log if e["id"] in {r["id"] for r in requests[:20]}]
print(f"\n{sum(e['ok'] for e in pilot)}/{len(pilot)} passed | attempts {statistics.mean(len(e['attempts'] or []) for e in pilot):.1f} on average"
      f" | {statistics.median(e['total_s'] for e in pilot):.0f} s per request (median) | wall clock {time.time() - t0:.0f} s")
print("why the failures failed:", [e["reasons"][:1] for e in pilot if not e["ok"]])

In [ ]:
# @title Full Gemini run (resumable; ~1.5-2 h at one thread; try THREADS = 3 if the pilot looked fine)
THREADS = 1  # @param {type:"integer"}
res = add_gemini(ds, requests, llm, threads=THREADS)
print(report(ds))

In [ ]:
# @title Procedural motions (seeds 0-239) and the course sit
!python -m kflib.generate procedural --dataset "{DATASET}" --n 240 --workers {os.cpu_count()} 2>/dev/null | tail -2
!python -m kflib.generate course --dataset "{DATASET}" 2>/dev/null | tail -1

In [ ]:
# @title Report: is the data broad, not one kind of motion?
import json
from IPython.display import Image, display
print(json.dumps(report(ds, DATASET / "report.png"), indent=1))
display(Image(str(DATASET / "report.png")))

In [ ]:
# @title A 10-motion library for Phase 3's smoke test
from kflib.dataset import build_library
rows = [r for r in ds.rows() if r["split"] == "train" and not r["aug"]]
pick = {r["id"]: ds.load_motion(r["id"]) for r in rows[::max(1, len(rows) // 10)][:10]}
lib = build_library(pick, DATASET / "library_smoke10.npz")
print(f"{len(pick)} motions, {len(lib.root_pos) / 50:.0f} s ->", DATASET / "library_smoke10.npz")